# EOD Zootopia — Analyse Comparative Dry / Normal / Wet
### Sujet 1 : Modélisation de l'Hydraulique
**Nathan Grimmer · Léon Le Goïc · Guillaume Reynier** — ETE305, ISAE-SUPAERO 2024-2025

---
Ce notebook charge les résultats des trois scénarios (CSV exportés par le code Julia) et produit l'ensemble des graphiques d'analyse.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from openpyxl import load_workbook
import warnings
warnings.filterwarnings('ignore')

# ── Style global ────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':       'Arial',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'figure.dpi':        120,
})

COLORS   = {'dry': '#E74C3C', 'normal': '#3498DB', 'wet': '#27AE60'}
LABELS   = {'dry': 'DRY (×0.6)', 'normal': 'NORMAL (×1.0)', 'wet': 'WET (×1.4)'}
SCENARIOS = ['dry', 'normal', 'wet']

# ── Chargement des CSV ──────────────────────────────────────────────────────
results = {}
for sc in SCENARIOS:
    try:
        results[sc] = pd.read_csv(f'../results/results_{sc}.csv')
        print(f'  {sc.upper():6} : {len(results[sc]):5d} heures chargées')
    except FileNotFoundError:
        print(f'  {sc.upper():6} : fichier non trouvé — relancer la simulation Julia')

print(f'\nColonnes : {list(results[SCENARIOS[0]].columns) if results else "N/A"}')

---
## 0. Chargement des limites historiques et du plancher interpolé

In [ ]:
# ── Limites historiques (Excel) ─────────────────────────────────────────────
excel_file = '../data/Donnees_etude_de_cas_ETE305.xlsx'
wb = load_workbook(excel_file)
ws = wb['Stock hydro']

lev_low, lev_high = [], []
for row in range(4, 369):
    lev_low.append(ws[f'B{row}'].value)
    lev_high.append(ws[f'C{row}'].value)

shift_days = 181   # même valeur que HYDRO_SHIFT_DAYS dans Julia
lev_low    = np.roll(np.array(lev_low,  dtype=float), -shift_days)
lev_high   = np.roll(np.array(lev_high, dtype=float), -shift_days)

stock_min_daily = (lev_low  / 100) * 1.0   # TWh
stock_max_daily = (lev_high / 100) * 1.0

# Étendre sur 8760h
STOCK_MIN_H = np.repeat(stock_min_daily, 24)[:8760]
STOCK_MAX_H = np.repeat(stock_max_daily, 24)[:8760]
HOURS       = np.arange(8760)

# ── Plancher interpolé (identique au code Julia) ─────────────────────────────
STOCK_FLOOR_ANCHORS = [
    (1,      0.52),
    (2160,   0.54),   # fin mars
    (3624,   0.58),   # fin mai
    (6552,   0.34),   # fin septembre
    (8040,   0.52),   # fin novembre
    (8760,   0.52),
]

def stock_floor(hour_1indexed):
    a = STOCK_FLOOR_ANCHORS
    if hour_1indexed <= a[0][0]:  return a[0][1]
    if hour_1indexed >= a[-1][0]: return a[-1][1]
    for i in range(len(a) - 1):
        h0, v0 = a[i]; h1, v1 = a[i+1]
        if h0 <= hour_1indexed <= h1:
            return v0 + (v1 - v0) * (hour_1indexed - h0) / (h1 - h0)

FLOOR_CURVE = np.array([stock_floor(h+1) for h in range(8760)])

print('Limites historiques et plancher interpolé prêts.')
print(f'  Stock min hist. : {STOCK_MIN_H.min():.2f} – {STOCK_MIN_H.max():.2f} TWh')
print(f'  Stock max hist. : {STOCK_MAX_H.min():.2f} – {STOCK_MAX_H.max():.2f} TWh')
print(f'  Plancher        : {FLOOR_CURVE.min():.2f} – {FLOOR_CURVE.max():.2f} TWh')

---
## 1. Résumé annuel — tableau de bord

In [ ]:
summary = {}
for sc in SCENARIOS:
    if sc not in results: continue
    df = results[sc]
    s  = {
        'charge':       df['load'].sum()         / 1e6,
        'hydro_lac':    df['Phy_lac'].sum()       / 1e6,
        'hydro_fdl':    df['Phy_fdl'].sum()       / 1e6,
        'P_nuc':        df['P_nuc'].sum()         / 1e6,
        'P_charbon':    df['P_charbon'].sum()     / 1e6,
        'P_ccg':        df['P_ccg'].sum()         / 1e6,
        'P_tac':        df['P_tac'].sum()         / 1e6,
        'P_cogen':      df['P_cogen'].sum()       / 1e6,
        'P_fioul':      df['P_fioul'].sum()       / 1e6,
        'P_eolien':     df['P_eolien'].sum()      / 1e6,
        'P_solaire':    df['P_solaire'].sum()     / 1e6,
        'P_dechets':    df['P_dechets'].sum()     / 1e6,
        'P_biomasse':   df['P_biomasse'].sum()    / 1e6,
        'step_turb':    df['Pdecharge_STEP'].sum()/ 1e6,
        'step_pump':    df['Pcharge_STEP'].sum()  / 1e6,
        'spill':        df['Pspill'].sum()        / 1e6,
        'puns':         df['Puns'].sum()          / 1e6,
        'stock_mean':   df['stock_hydro'].mean()  / 1e6,
        'stock_min':    df['stock_hydro'].min()   / 1e6,
        'stock_max':    df['stock_hydro'].max()   / 1e6,
        'water_px_mean':df['water_price'].mean()  if 'water_price' in df else np.nan,
        'v_floor':      (df['stock_hydro'].values/1e6 < FLOOR_CURVE[:len(df)]).sum(),
        'v_min_hist':   (df['stock_hydro'].values/1e6 < STOCK_MIN_H[:len(df)]).sum(),
        'v_max_hist':   (df['stock_hydro'].values/1e6 > STOCK_MAX_H[:len(df)]).sum(),
    }
    s['hydro_total'] = s['hydro_lac'] + s['hydro_fdl']
    s['thermique']   = s['P_charbon'] + s['P_ccg'] + s['P_tac'] + s['P_cogen'] + s['P_fioul']
    summary[sc] = s

print(f'{"":20} {"DRY":>10} {"NORMAL":>10} {"WET":>10}   unité')
print('─' * 65)
rows = [
    ('Charge totale',      'charge',       'TWh'),
    ('Hydraulique lacs',   'hydro_lac',    'TWh'),
    ('Hydraulique FDL',    'hydro_fdl',    'TWh'),
    ('Nucléaire',          'P_nuc',        'TWh'),
    ('Charbon',            'P_charbon',    'TWh'),
    ('Gaz CCG',            'P_ccg',        'TWh'),
    ('Gaz TAC',            'P_tac',        'TWh'),
    ('Éolien + Solaire',   'P_eolien',     'TWh'),
    ('STEP turbinage',     'step_turb',    'TWh'),
    ('STEP pompage',       'step_pump',    'TWh'),
    ('Spill hydro',        'spill',        'TWh'),
    ('Défaillance (ENF)',  'puns',         'TWh'),
    ('Stock moyen',        'stock_mean',   'TWh'),
    ('Prix eau moyen',     'water_px_mean','€/MWh'),
    ('Viol. plancher',     'v_floor',      'h'),
    ('Viol. min hist.',    'v_min_hist',   'h'),
]
for label, key, unit in rows:
    vals = [summary[sc][key] if sc in summary else float('nan') for sc in SCENARIOS]
    fmt  = '{:10.0f}' if unit == 'h' else '{:10.2f}'
    print(f'{label:20} ' + ''.join(fmt.format(v) for v in vals) + f'   {unit}')

---
## 2. Mix de production annuel — barres empilées

In [ ]:
filières = [
    ('Nucléaire',    'P_nuc',     '#1565C0'),
    ('Charbon',      'P_charbon', '#5D4037'),
    ('Gaz CCG',      'P_ccg',     '#F57C00'),
    ('Gaz TAC',      'P_tac',     '#FFB300'),
    ('Cogén. + Fioul','P_fioul',  '#9E9E9E'),
    ('Éolien',       'P_eolien',  '#00ACC1'),
    ('Solaire',      'P_solaire', '#FDD835'),
    ('Hydro lacs',   'Phy_lac',   '#1B5E20'),
    ('Hydro FDL',    'Phy_fdl',   '#A5D6A7'),
    ('STEP turb.',   'Pdecharge_STEP', '#6A1B9A'),
    ('Déchets+Bio',  'P_dechets', '#78909C'),
]

fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={'width_ratios': [2, 1]})

# ── Barres empilées ─────────────────────────────────────────────────────────
ax = axes[0]
x  = np.arange(3)
bottoms = np.zeros(3)
for label, col, color in filières:
    vals = []
    for sc in SCENARIOS:
        if sc not in results:
            vals.append(0)
        else:
            c2 = 'P_cogen' if col == 'P_fioul' else col
            v  = results[sc][col].sum() / 1e6
            if col == 'P_fioul':
                v += results[sc]['P_cogen'].sum() / 1e6
            vals.append(v)
    vals = np.array(vals)
    ax.bar(x, vals, bottom=bottoms, color=color, label=label, edgecolor='white', linewidth=0.5)
    bottoms += vals

ax.set_xticks(x)
ax.set_xticklabels([LABELS[sc] for sc in SCENARIOS], fontsize=11)
ax.set_ylabel('Production annuelle (TWh)', fontsize=12)
ax.set_title('Mix de production annuel par scénario', fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=8, bbox_to_anchor=(1.01, 1))

# ── Part hydraulique (camembert comparatif) ─────────────────────────────────
ax2 = axes[1]
hydro_parts = []
thermal_parts = []
nuc_parts = []
enr_parts = []
for sc in SCENARIOS:
    if sc not in summary: continue
    s = summary[sc]
    total = s['charge']
    hydro_parts.append(100 * s['hydro_total'] / total)
    thermal_parts.append(100 * s['thermique'] / total)
    nuc_parts.append(100 * s['P_nuc'] / total)
    enr_parts.append(100 * (s['P_eolien'] + s['P_solaire'] + s['P_dechets'] + s['P_biomasse']) / total)

sc_labels = [LABELS[sc] for sc in SCENARIOS if sc in summary]
bw = 0.2
xi = np.arange(len(sc_labels))
ax2.bar(xi - bw*1.5, nuc_parts,     bw, color='#1565C0', label='Nucléaire')
ax2.bar(xi - bw*0.5, hydro_parts,   bw, color='#1B5E20', label='Hydraulique')
ax2.bar(xi + bw*0.5, enr_parts,     bw, color='#00ACC1', label='ENR + Bio')
ax2.bar(xi + bw*1.5, thermal_parts, bw, color='#F57C00', label='Thermique flamme')
ax2.set_xticks(xi)
ax2.set_xticklabels(sc_labels, fontsize=9)
ax2.set_ylabel('Part dans la charge (%)', fontsize=11)
ax2.set_title('Décomposition par filière (%)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../results/fig1_mix_production.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Évolution du stock hydraulique — comparaison 3 scénarios

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=True)

ANCHOR_STYLES = {
    2160: {'color': '#2196F3', 'label': 'Fin mars'},
    3624: {'color': '#4CAF50', 'label': 'Fin mai'},
    6552: {'color': '#FF9800', 'label': 'Fin sept.'},
    8040: {'color': '#9C27B0', 'label': 'Fin nov.'},
}

for idx, sc in enumerate(SCENARIOS):
    if sc not in results: continue
    df    = results[sc]
    ax    = axes[idx]
    stock = df['stock_hydro'].values / 1e6
    hrs   = np.arange(len(stock))
    n     = len(stock)

    s_min = STOCK_MIN_H[:n]
    s_max = STOCK_MAX_H[:n]
    floor = FLOOR_CURVE[:n]

    # Zone historique
    ax.fill_between(hrs, s_min, s_max, alpha=0.12, color='green', label='Zone historique')
    ax.plot(hrs, s_min, 'g-',       lw=1.0, alpha=0.5)
    ax.plot(hrs, s_max, color='darkgreen', lw=1.0, alpha=0.5)

    # Plancher interpolé
    ax.plot(hrs, floor, 'k--', lw=2.0, label='Plancher', zorder=5)
    ax.fill_between(hrs, 0, floor, alpha=0.05, color='black')

    # Violations du plancher
    viol = stock < floor
    if viol.any():
        ax.fill_between(hrs, stock, floor, where=viol, alpha=0.35, color='red', label=f'Viol. plancher ({viol.sum()}h)')

    # Stock réel
    ax.fill_between(hrs, 0, stock, alpha=0.20, color=COLORS[sc])
    ax.plot(hrs, stock, color=COLORS[sc], lw=1.5, label='Stock réel')

    # Points d'ancrage
    for h_anc, sty in ANCHOR_STYLES.items():
        tgt = stock_floor(h_anc)
        hi  = min(h_anc - 1, n - 1)
        ax.scatter(hi, tgt, color=sty['color'], s=60, zorder=6)
        ax.annotate(f"{tgt:.2f}", xy=(hi, tgt), xytext=(0, 7),
                    textcoords='offset points', ha='center',
                    fontsize=7.5, color=sty['color'], fontweight='bold')

    ax.set_title(f'{LABELS[sc]}\nMin:{stock.min():.2f} | Moy:{stock.mean():.2f} | Max:{stock.max():.2f} TWh',
                 fontweight='bold', fontsize=11)
    ax.set_xlabel("Heure de l'année", fontsize=10)
    if idx == 0: ax.set_ylabel('Stock (TWh)', fontsize=11)
    ax.legend(fontsize=7)
    ax.set_xlim(0, n)
    ax.set_ylim(bottom=0)

plt.suptitle('Évolution du Stock Hydraulique — Limites Historiques & Plancher Interpolé',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/fig2_stock_hydraulique.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Stock hydraulique — superposition des 3 scénarios

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.fill_between(HOURS, STOCK_MIN_H, STOCK_MAX_H, alpha=0.10, color='green', label='Zone historique')
ax.plot(HOURS, FLOOR_CURVE, 'k--', lw=2, label='Plancher interpolé', zorder=5)

for sc in SCENARIOS:
    if sc not in results: continue
    stock = results[sc]['stock_hydro'].values / 1e6
    n     = len(stock)
    ax.plot(np.arange(n), stock, color=COLORS[sc], lw=1.5, alpha=0.85, label=LABELS[sc])

# Annotations saisonnières
seasons = [
    (2160, 'Fin mars'),
    (3624, 'Fin mai'),
    (6552, 'Fin sept.'),
    (8040, 'Fin nov.'),
]
for h, lbl in seasons:
    ax.axvline(h, color='gray', lw=0.8, ls=':', alpha=0.6)
    ax.text(h + 30, ax.get_ylim()[1] * 0.97 if ax.get_ylim()[1] > 0 else 0.9,
            lbl, fontsize=8, color='gray', rotation=90, va='top')

ax.set_xlabel("Heure de l'année", fontsize=12)
ax.set_ylabel('Stock hydraulique (TWh)', fontsize=12)
ax.set_title('Superposition des trajectoires de stock — 3 scénarios', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(0, 8760)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig('../results/fig3_stock_superpose.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Prix de l'eau et signal économique

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)

# ── Panneau haut : stock hydraulique ────────────────────────────────────────
ax1 = axes[0]
for sc in SCENARIOS:
    if sc not in results: continue
    stock = results[sc]['stock_hydro'].values / 1e6
    ax1.plot(np.arange(len(stock)), stock, color=COLORS[sc], lw=1.3, label=LABELS[sc])
ax1.plot(HOURS, FLOOR_CURVE, 'k--', lw=1.5, label='Plancher')
ax1.set_ylabel('Stock (TWh)', fontsize=11)
ax1.set_title('Stock hydraulique et prix de l\'eau au cours de l\'année', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)

# Seuils de prix
for px, lbl, col in [(12, 'Nuc. (12)', '#1565C0'), (40, 'CCG (40)', '#F57C00'), (70, 'TAC (70)', '#C62828')]:
    # Convertir prix → stock correspondant (inversion de water_price_from_stock)
    pass  # annotations sur le panneau bas

# ── Panneau bas : prix de l'eau ──────────────────────────────────────────────
ax2 = axes[1]
for sc in SCENARIOS:
    if sc not in results or 'water_price' not in results[sc]: continue
    px = results[sc]['water_price'].values
    # Moyenne glissante 168h pour lisibilité
    px_smooth = pd.Series(px).rolling(168, center=True).mean().values
    ax2.plot(np.arange(len(px)), px, color=COLORS[sc], lw=0.5, alpha=0.3)
    ax2.plot(np.arange(len(px_smooth)), px_smooth, color=COLORS[sc], lw=2.0, label=f'{LABELS[sc]} (moy. 7j)')

for px_ref, lbl, col in [(12, 'Nucléaire 12 €', '#1565C0'), (40, 'CCG 40 €', '#F57C00'), (70, 'TAC 70 €', '#C62828')]:
    ax2.axhline(px_ref, ls='--', color=col, lw=1.2, alpha=0.7)
    ax2.text(100, px_ref + 1, lbl, fontsize=8, color=col)

ax2.set_ylabel('Prix de l\'eau (€/MWh)', fontsize=11)
ax2.set_xlabel("Heure de l'année", fontsize=11)
ax2.legend(fontsize=9)
ax2.set_ylim(0, 85)
ax2.set_xlim(0, 8760)

plt.tight_layout()
plt.savefig('../results/fig4_prix_eau.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Gestion de la STEP — stock et cycles charge/décharge

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 12), sharex='col')

for row, sc in enumerate(SCENARIOS):
    if sc not in results: continue
    df    = results[sc]
    hrs   = np.arange(len(df))
    stock = df['stock_STEP'].values / 1e6
    charge   = df['Pcharge_STEP'].values / 1e3    # GW
    decharge = df['Pdecharge_STEP'].values / 1e3  # GW

    # Stock STEP
    ax = axes[row, 0]
    ax.fill_between(hrs, 0, stock, alpha=0.3, color=COLORS[sc])
    ax.plot(hrs, stock, color=COLORS[sc], lw=1.2)
    ax.axhline(1.0, ls='--', color='gray', lw=1, alpha=0.6, label='Capacité max (1 TWh)')
    ax.set_ylabel(f'{LABELS[sc]}\nStock STEP (TWh)', fontsize=9)
    ax.set_ylim(0, 1.05)
    if row == 0: ax.set_title('Stock STEP', fontsize=11, fontweight='bold')

    # Charge / décharge (moyenne 24h)
    ax2 = axes[row, 1]
    ch_s = pd.Series(charge).rolling(24, center=True).mean().values
    de_s = pd.Series(decharge).rolling(24, center=True).mean().values
    ax2.fill_between(hrs,  0,  ch_s, alpha=0.4, color='steelblue', label='Pompage (moy. 24h)')
    ax2.fill_between(hrs,  0, -de_s, alpha=0.4, color='coral',     label='Turbinage (moy. 24h)')
    ax2.axhline(0, color='black', lw=0.8)
    ax2.set_ylabel('Puissance (GW)', fontsize=9)
    if row == 0:
        ax2.set_title('Charge / Décharge STEP', fontsize=11, fontweight='bold')
        ax2.legend(fontsize=8, loc='upper right')

    if row == 2:
        axes[row, 0].set_xlabel("Heure de l'année", fontsize=10)
        axes[row, 1].set_xlabel("Heure de l'année", fontsize=10)

plt.suptitle('Gestion de la STEP Polochon — 3 scénarios', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/fig5_STEP.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Recours au thermique à flamme

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Durée de fonctionnement annuelle ────────────────────────────────────────
thermal_units = [
    ('Nucléaire',  'UC_nuc',    '#1565C0'),
    ('Charbon',    'UC_charbon','#5D4037'),
    ('Gaz CCG',    'UC_ccg',    '#F57C00'),
    ('Gaz TAC',    'UC_tac',    '#FFB300'),
    ('Cogénération','UC_cogen', '#9E9E9E'),
    ('Fioul',      'UC_fioul',  '#78909C'),
]
ax1 = axes[0]
x   = np.arange(len(thermal_units))
bw  = 0.25
for i, sc in enumerate(SCENARIOS):
    if sc not in results: continue
    df   = results[sc]
    vals = [df[col].sum() if col in df.columns else 0 for _, col, _ in thermal_units]
    ax1.bar(x + (i - 1) * bw, vals, bw, color=COLORS[sc], label=LABELS[sc], alpha=0.85)

ax1.set_xticks(x)
ax1.set_xticklabels([l for l, _, _ in thermal_units], fontsize=10)
ax1.set_ylabel('Heures × machines / an', fontsize=11)
ax1.set_title('Durée de fonctionnement des unités thermiques', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)

# ── Production thermique mensuelle DRY vs WET ─────────────────────────────
ax2 = axes[1]
months = [f'M{m}' for m in range(1, 13)]
HOURS_CUM = [0, 744, 744+672, 744+672+744, 744+672+744+720,
             744+672+744+720+744, 744+672+744+720+744+720,
             744+672+744+720+744+720+744, 744+672+744+720+744+720+744+744,
             744+672+744+720+744+720+744+744+720,
             744+672+744+720+744+720+744+744+720+744,
             744+672+744+720+744+720+744+744+720+744+720, 8760]

for sc in ['dry', 'wet']:
    if sc not in results: continue
    df   = results[sc]
    therm_col = ['P_charbon', 'P_ccg', 'P_tac', 'P_cogen', 'P_fioul']
    therm = sum(df[c] for c in therm_col if c in df.columns).values / 1e6
    monthly = [therm[HOURS_CUM[m]:HOURS_CUM[m+1]].sum() for m in range(12)]
    ax2.plot(months, monthly, marker='o', color=COLORS[sc], lw=2, label=LABELS[sc])

ax2.set_ylabel('Production thermique (TWh)', fontsize=11)
ax2.set_title('Production thermique mensuelle — DRY vs WET', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.tick_params(axis='x', labelsize=9)

plt.tight_layout()
plt.savefig('../results/fig6_thermique.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Production hydraulique — décomposition mensuelle

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
MOIS = ['Jan','Fév','Mar','Avr','Mai','Jun','Jul','Aoû','Sep','Oct','Nov','Déc']

for ax, col, title in zip(axes, ['Phy_lac', 'Phy_fdl'], ['Lacs de retenue', 'Fil de l\'eau']):
    for sc in SCENARIOS:
        if sc not in results: continue
        vals = results[sc][col].values / 1e6
        monthly = [vals[HOURS_CUM[m]:HOURS_CUM[m+1]].sum() for m in range(12)]
        ax.plot(MOIS, monthly, marker='o', color=COLORS[sc], lw=2, label=LABELS[sc])
    ax.set_title(f'Production hydraulique mensuelle — {title}', fontsize=12, fontweight='bold')
    ax.set_ylabel('Production (TWh/mois)', fontsize=11)
    ax.legend(fontsize=9)
    ax.tick_params(axis='x', labelsize=9)

plt.tight_layout()
plt.savefig('../results/fig7_hydro_mensuel.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Analyse des violations de contraintes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

viol_types = [
    ('Violations min hist.', 'v_min_hist', STOCK_MIN_H),
    ('Violations max hist.', 'v_max_hist', STOCK_MAX_H),
    ('Violations plancher',  'v_floor',    FLOOR_CURVE),
]

for ax, (title, key, ref) in zip(axes, viol_types):
    for sc in SCENARIOS:
        if sc not in results: continue
        stock = results[sc]['stock_hydro'].values / 1e6
        n     = len(stock)
        ref_n = ref[:n]
        if key == 'v_min_hist':  mask = stock < ref_n
        elif key == 'v_max_hist': mask = stock > ref_n
        else:                     mask = stock < ref_n

        # Profil temporel des violations
        ax.fill_between(np.arange(n), mask.astype(float),
                        alpha=0.4, color=COLORS[sc], label=f'{LABELS[sc]} ({mask.sum()}h)')

    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel("Heure de l'année", fontsize=10)
    ax.set_ylabel('Violation (0/1)', fontsize=10)
    ax.set_ylim(-0.05, 1.2)
    ax.legend(fontsize=8)

plt.suptitle('Profil temporel des violations de contraintes hydrauliques', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/fig8_violations.png', dpi=150, bbox_inches='tight')
plt.show()

# Résumé violations
print('\n' + '='*70)
print('RÉSUMÉ DES VIOLATIONS')
print('='*70)
for sc in SCENARIOS:
    if sc not in results: continue
    stock = results[sc]['stock_hydro'].values / 1e6
    n     = len(stock)
    v_min  = (stock < STOCK_MIN_H[:n]).sum()
    v_max  = (stock > STOCK_MAX_H[:n]).sum()
    v_fl   = (stock < FLOOR_CURVE[:n]).sum()
    slack  = results[sc]['slack_stock_min'].sum() if 'slack_stock_min' in results[sc] else float('nan')
    print(f'  {sc.upper():6} | Viol. min hist: {v_min:5d}h | Viol. max hist: {v_max:5d}h | Viol. plancher: {v_fl:5d}h | Slack total: {slack/1e6:.4f} TWh')

---
## 10. Analyse hebdomadaire type — zoom sur une semaine représentative

In [ ]:
# Semaine estivale (heure 4320 = début juillet) — période de creux hydraulique
H_START = 4320
H_END   = H_START + 168
JOURS   = ['Lun', 'Mar', 'Mer', 'Jeu', 'Ven', 'Sam', 'Dim']

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
cols_plot = [
    ('Charge vs Production',   None),
    ('Hydraulique',            None),
    ('Thermique (CCG+TAC)',    None),
]

for row, sc in enumerate(SCENARIOS):
    if sc not in results: continue
    df   = results[sc].iloc[H_START:H_END]
    hrs  = np.arange(len(df))
    tick_pos = np.arange(0, 168, 24)

    # Col 0 : bilan offre-demande
    ax = axes[row, 0]
    ax.stackplot(hrs,
        df['P_nuc'].values/1e3,
        df['P_charbon'].values/1e3,
        (df['P_ccg'] + df['P_tac']).values/1e3,
        df['Phy_lac'].values/1e3,
        df['Phy_fdl'].values/1e3,
        df['Pdecharge_STEP'].values/1e3,
        (df['P_eolien']+df['P_solaire']).values/1e3,
        colors=['#1565C0','#5D4037','#F57C00','#1B5E20','#A5D6A7','#6A1B9A','#00ACC1'],
        labels=['Nuc.','Charbon','Gaz','Hydro lac','FDL','STEP','ENR']
    )
    ax.plot(hrs, df['load'].values/1e3, 'k-', lw=2, label='Charge')
    ax.set_title(f'{LABELS[sc]} — Bilan', fontsize=10, fontweight='bold')
    ax.set_ylabel('GW', fontsize=9)
    ax.set_xticks(tick_pos); ax.set_xticklabels(JOURS, fontsize=8)
    if row == 0: ax.legend(fontsize=6, ncol=2, loc='upper right')

    # Col 1 : stock hydraulique
    ax2 = axes[row, 1]
    ax2.fill_between(hrs, df['stock_hydro'].values/1e6, alpha=0.3, color=COLORS[sc])
    ax2.plot(hrs, df['stock_hydro'].values/1e6, color=COLORS[sc], lw=2)
    ax2.plot(hrs, FLOOR_CURVE[H_START:H_START+len(df)], 'k--', lw=1.5, label='Plancher')
    ax2.set_title('Stock hydraulique (TWh)', fontsize=10, fontweight='bold')
    ax2.set_ylabel('TWh', fontsize=9)
    ax2.set_xticks(tick_pos); ax2.set_xticklabels(JOURS, fontsize=8)
    ax2.legend(fontsize=8)

    # Col 2 : STEP charge/décharge
    ax3 = axes[row, 2]
    ax3.fill_between(hrs,  df['Pcharge_STEP'].values/1e3,   0, alpha=0.5, color='steelblue', label='Pompage')
    ax3.fill_between(hrs, -df['Pdecharge_STEP'].values/1e3, 0, alpha=0.5, color='coral',     label='Turbinage')
    ax3.axhline(0, color='black', lw=0.8)
    ax3_twin = ax3.twinx()
    ax3_twin.plot(hrs, df['stock_STEP'].values/1e6, 'm-', lw=1.5, alpha=0.7, label='Stock STEP')
    ax3_twin.set_ylabel('Stock STEP (TWh)', fontsize=8, color='m')
    ax3.set_title('STEP Polochon', fontsize=10, fontweight='bold')
    ax3.set_ylabel('GW', fontsize=9)
    ax3.set_xticks(tick_pos); ax3.set_xticklabels(JOURS, fontsize=8)
    ax3.legend(fontsize=8, loc='upper left')
    ax3_twin.legend(fontsize=8, loc='upper right')

plt.suptitle(f'Zoom semaine estivale (h={H_START}–{H_END}) — période de tension hydraulique',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/fig9_zoom_semaine.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 11. Synthèse économique — coûts opérationnels estimés

In [ ]:
PRIX_MARCHE = {
    'P_nuc':    12, 'P_charbon': 36, 'P_ccg': 40, 'P_tac': 70,
    'P_cogen':  70, 'P_fioul':  100,
}
COST_SPILL = 8000

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Coûts par filière ────────────────────────────────────────────────────────
filière_costs = list(PRIX_MARCHE.keys()) + ['hydro', 'spill', 'STEP']
cost_labels   = ['Nucl.', 'Charbon', 'CCG', 'TAC', 'Cogén.', 'Fioul', 'Hydro', 'Spill', 'STEP']
cost_colors   = ['#1565C0','#5D4037','#F57C00','#FFB300','#9E9E9E','#78909C','#1B5E20','red','#6A1B9A']

cost_data = {sc: [] for sc in SCENARIOS}
for sc in SCENARIOS:
    if sc not in results: continue
    df = results[sc]
    for col, prix in PRIX_MARCHE.items():
        c = df[col].sum() * prix / 1e9 if col in df.columns else 0
        cost_data[sc].append(c)
    # Hydro : prix eau moyen × production
    if 'water_price' in df.columns:
        hydro_cost = (df['water_price'] * (df['Phy_lac'] + df['Phy_fdl'])).sum() / 1e9
    else:
        hydro_cost = 0
    cost_data[sc].append(hydro_cost)
    cost_data[sc].append(df['Pspill'].sum() * COST_SPILL / 1e9 if 'Pspill' in df.columns else 0)
    cost_data[sc].append(df['Pcharge_STEP'].sum() * 1 / 1e9 if 'Pcharge_STEP' in df.columns else 0)

ax1 = axes[0]
x   = np.arange(len(filière_costs))
bw  = 0.25
for i, sc in enumerate(SCENARIOS):
    if sc not in cost_data or not cost_data[sc]: continue
    ax1.bar(x + (i-1)*bw, cost_data[sc], bw, color=COLORS[sc], label=LABELS[sc], alpha=0.85)

ax1.set_xticks(x); ax1.set_xticklabels(cost_labels, fontsize=9)
ax1.set_ylabel('Coût annuel (G€)', fontsize=11)
ax1.set_title('Coûts opérationnels par filière', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)

# ── Coût total ───────────────────────────────────────────────────────────────
ax2 = axes[1]
totals = {sc: sum(cost_data[sc]) for sc in SCENARIOS if sc in cost_data and cost_data[sc]}
bars   = ax2.bar([LABELS[sc] for sc in SCENARIOS if sc in totals],
                 [totals[sc] for sc in SCENARIOS if sc in totals],
                 color=[COLORS[sc] for sc in SCENARIOS if sc in totals],
                 edgecolor='white', alpha=0.85, width=0.5)
for bar, sc in zip(bars, [s for s in SCENARIOS if s in totals]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{totals[sc]:.2f} G€', ha='center', fontsize=10, fontweight='bold')
ax2.set_ylabel('Coût total annuel (G€)', fontsize=11)
ax2.set_title('Coût opérationnel total — 3 scénarios', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/fig10_couts.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nCoûts totaux estimés :')
for sc in SCENARIOS:
    if sc in totals:
        print(f'  {sc.upper():6} : {totals[sc]:.3f} G€/an')

---
## 12. Export de tous les graphiques
*(Toutes les figures ont déjà été sauvegardées en PNG dans `../results/`)*

In [ ]:
import os
figs = [f for f in os.listdir('../results/') if f.startswith('fig') and f.endswith('.png')]
print('Figures exportées :')
for f in sorted(figs):
    size = os.path.getsize(f'../results/{f}') // 1024
    print(f'  {f:40s} {size:6d} Ko')